# Applied Data Mining with Python

## Part 3: Advanced Models & Benchmarking

*Data Science Workshop Series*  ·  Notebook companion to the Quarto slide deck.

> This notebook runs top-to-bottom on Google Colab's free tier. Each **Hands-on Exercise** has a `# TODO` cell to complete, followed by a collapsible **💡 Show solution**. Run the setup cells in order first.

In [ ]:
# --- Robust data loader (added for the notebook version) -------------------------------
# Loads the wine CSV from the UCI repository, falling back to a GitHub mirror if UCI is
# unreachable during a live session. Used everywhere the deck wrote pd.read_csv(url, ...).
import pandas as pd

def load_wine(url, sep=";"):
    try:
        return pd.read_csv(url, sep=sep)
    except Exception as e:
        fname = url.rsplit("/", 1)[-1]
        mirror = "https://raw.githubusercontent.com/zygmuntz/wine-quality/master/winequality/" + fname
        print(f"Primary source failed ({type(e).__name__}); falling back to mirror.")
        return pd.read_csv(mirror, sep=sep)

## Part 3 Overview

**Advanced Models & Benchmarking** (≈ 3 hours)

| Module | Topic | Duration |
|--------|-------|----------|
| 3.1 | Model Evaluation Metrics Deep Dive | 35 min |
| 3.2 | Cross-Validation Strategies | 30 min |
| 3.3 | Introduction to XGBoost | 40 min |
| 3.4 | Model Comparison Framework | 30 min |
| 3.5 | Final Benchmarking & Leaderboard | 25 min |
| 3.6 | Course Conclusion & Best Practices | 20 min |

## Learning Objectives

By the end of this session, you will be able to:

- Select appropriate evaluation metrics for different problem types
- Implement robust cross-validation workflows
- Train and tune XGBoost models effectively
- Build a systematic model comparison framework
- Produce production-quality benchmark reports
- Apply best practices from the complete data mining lifecycle

## Setup: Loading Data and Previous Work

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error,
                             accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix)
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Load wine quality dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine_df = load_wine(url)

# Create engineered features from Part 2
wine_df['free_sulfur_ratio'] = wine_df['free sulfur dioxide'] / (wine_df['total sulfur dioxide'] + 0.001)
wine_df['acid_ratio'] = wine_df['fixed acidity'] / (wine_df['volatile acidity'] + 0.001)
wine_df['alcohol_density'] = wine_df['alcohol'] * wine_df['density']
wine_df['log_chlorides'] = np.log1p(wine_df['chlorides'])

# Binary classification target
wine_df['good_wine'] = (wine_df['quality'] >= 7).astype(int)

print(f"Dataset loaded with engineered features: {wine_df.shape}")
print(f"Target distribution (quality): {wine_df['quality'].min()}-{wine_df['quality'].max()}")
print(f"Good wine percentage: {wine_df['good_wine'].mean()*100:.1f}%")

# Module 3.1: Model Evaluation Metrics Deep Dive

## Conceptual Overview

Choosing the right metric is critical because different metrics measure different aspects of model performance:

**Regression metrics**: How close predictions are to actual values

**Classification metrics**: How well the model separates classes

The best metric depends on the business problem and cost of different types of errors.

## Wow Factor Hook

You will see how a model that looks excellent by one metric can be poor by another — and understand when each metric matters.

## Code: Regression Metrics

In [ ]:
from sklearn.metrics import (mean_squared_error, mean_absolute_error, 
                             r2_score, mean_absolute_percentage_error)

# Prepare regression data
X = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['quality']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = Ridge(alpha=1.0)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

# Calculate all regression metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print("REGRESSION METRICS COMPARISON")
print("=" * 50)
print(f"\nMean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R² Score: {r2:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2%}")

print("\nMetric Interpretations:")
print(f"  - RMSE of {rmse:.2f} means predictions are off by ~{rmse:.2f} quality points on average")
print(f"  - MAE of {mae:.2f} is the average absolute error")
print(f"  - R² of {r2:.2f} means {r2*100:.1f}% of variance is explained")

## Code: When Metrics Disagree

In [ ]:
# Demonstrate how outlier predictions affect different metrics
np.random.seed(42)

# Simulate two models with different error patterns
y_test_sample = np.array([5, 5, 6, 6, 7, 7, 5, 6, 5, 7])

# Model A: Small consistent errors
y_pred_a = np.array([5.3, 5.4, 5.8, 6.2, 6.7, 6.8, 5.2, 5.9, 5.1, 6.9])

# Model B: Mostly accurate but one big outlier
y_pred_b = np.array([5.0, 5.0, 6.0, 6.0, 7.0, 7.0, 5.0, 6.0, 5.0, 4.0])  # Last one is way off

print("TWO MODELS WITH DIFFERENT ERROR PATTERNS")
print("=" * 50)
print("\nModel A (consistent small errors):")
print(f"  MSE: {mean_squared_error(y_test_sample, y_pred_a):.4f}")
print(f"  MAE: {mean_absolute_error(y_test_sample, y_pred_a):.4f}")

print("\nModel B (one big outlier):")
print(f"  MSE: {mean_squared_error(y_test_sample, y_pred_b):.4f}")
print(f"  MAE: {mean_absolute_error(y_test_sample, y_pred_b):.4f}")

print("\nInsight:")
print("  - MSE penalizes large errors quadratically")
print("  - MAE treats all errors equally")
print("  - Model B has better MAE despite one large error")

## Code: Classification Metrics

In [ ]:
# Prepare classification data
X = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['good_wine']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_train_scaled, y_train)
y_pred = log_model.predict(X_test_scaled)
y_prob = log_model.predict_proba(X_test_scaled)[:, 1]

# All classification metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("CLASSIFICATION METRICS")
print("=" * 50)
print(f"\nAccuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

print("\nMetric Interpretations:")
print(f"  - Accuracy: {accuracy*100:.1f}% of all predictions are correct")
print(f"  - Precision: {precision*100:.1f}% of predicted 'good' wines are actually good")
print(f"  - Recall: {recall*100:.1f}% of actual 'good' wines are identified")
print(f"  - F1: Harmonic mean of precision and recall")
print(f"  - ROC-AUC: Probability that model ranks positive higher than negative")

## Code: Confusion Matrix Analysis

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, 
    display_labels=['Not Good', 'Good'],
    ax=axes[0],
    cmap='Blues'
)
axes[0].set_title('Confusion Matrix')

# ROC Curve
RocCurveDisplay.from_predictions(
    y_test, y_prob,
    ax=axes[1],
    name='Logistic Regression'
)
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_title('ROC Curve')
axes[1].legend()

# Precision-Recall Curve
PrecisionRecallDisplay.from_predictions(
    y_test, y_prob,
    ax=axes[2],
    name='Logistic Regression'
)
axes[2].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()

# Confusion matrix interpretation
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print("\nConfusion Matrix Breakdown:")
print(f"  True Negatives (correctly predicted Not Good): {tn}")
print(f"  False Positives (predicted Good, actually Not Good): {fp}")
print(f"  False Negatives (predicted Not Good, actually Good): {fn}")
print(f"  True Positives (correctly predicted Good): {tp}")

## Code: Choosing the Right Metric

In [ ]:
print("METRIC SELECTION GUIDE")
print("=" * 60)

print("""
REGRESSION:
┌─────────────────┬─────────────────────────────────────────┐
│ Metric          │ When to Use                             │
├─────────────────┼─────────────────────────────────────────┤
│ RMSE            │ When large errors are especially bad    │
│ MAE             │ When all errors matter equally          │
│ R²              │ For comparing models, explaining fit    │
│ MAPE            │ When percentage error matters           │
└─────────────────┴─────────────────────────────────────────┘

CLASSIFICATION:
┌─────────────────┬─────────────────────────────────────────┐
│ Metric          │ When to Use                             │
├─────────────────┼─────────────────────────────────────────┤
│ Accuracy        │ Balanced classes only                   │
│ Precision       │ When false positives are costly         │
│ Recall          │ When false negatives are costly         │
│ F1 Score        │ Balance between precision and recall    │
│ ROC-AUC         │ Overall ranking ability                 │
└─────────────────┴─────────────────────────────────────────┘

WINE QUALITY EXAMPLE:
- If we sell "premium wine" based on predictions:
  * High Precision: Avoid selling bad wine as premium
  * High Recall: Don't miss good wines for premium sales
""")

## Expected Output & Interpretation

Metric insights:

- **Accuracy ~87%** looks great but is misleading (majority class baseline is ~87%)
- **Precision ~65%** means 35% of "good" predictions are wrong
- **Recall ~45%** means we miss 55% of actual good wines
- **ROC-AUC ~0.82** shows decent overall discrimination ability

For imbalanced data, accuracy alone is insufficient.

## Hands-on Exercise 3.1

**Task**: Analyze metrics for a threshold-adjusted classifier.

1. Using the logistic regression predictions, vary the threshold from 0.3 to 0.7
2. For each threshold, calculate precision, recall, and F1 score
3. Plot all three metrics against threshold
4. At what threshold is F1 maximized?
5. What threshold would you choose if false positives cost twice as much as false negatives?

**Time**: 15 minutes

In [ ]:
# TODO: sweep thresholds 0.3..0.7 on y_prob; compute precision/recall/F1; plot; find F1-max threshold
# Hint: y_test and y_prob come from the Module 3.1 classification cell


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

# y_test, y_prob come from the Module 3.1 classification cell
ths = np.linspace(0.3, 0.7, 41)
P, R, F = [], [], []
for t in ths:
    pred = (y_prob >= t).astype(int)
    P.append(precision_score(y_test, pred, zero_division=0))
    R.append(recall_score(y_test, pred, zero_division=0))
    F.append(f1_score(y_test, pred, zero_division=0))

plt.plot(ths, P, label="Precision"); plt.plot(ths, R, label="Recall"); plt.plot(ths, F, label="F1")
plt.xlabel("Threshold"); plt.legend(); plt.title("Metrics vs threshold"); plt.show()

best = ths[int(np.argmax(F))]
print(f"F1 is maximized at threshold ~ {best:.3f}")
print("If false positives cost twice as much, push the threshold ABOVE the F1-optimal "
      "point to favor precision.")
```

</details>

# Module 3.2: Cross-Validation Strategies

## Conceptual Overview

A single train/test split provides only one estimate of model performance. Cross-validation provides multiple estimates by rotating which data serves as test set:

- **K-Fold CV**: Split data into K parts, train on K-1, test on 1, repeat K times
- **Stratified K-Fold**: Preserves class distribution in each fold
- **Leave-One-Out**: K = N (most thorough but computationally expensive)

Cross-validation gives more reliable performance estimates.

## Wow Factor Hook

You will see how cross-validation reveals performance variability that a single split hides — crucial for confident model selection.

## Code: Basic K-Fold Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

# Prepare data for regression
X = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['quality']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# K-Fold CV with Ridge regression
model = Ridge(alpha=1.0)

# 5-fold cross-validation
cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='r2')

print("5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 50)
print(f"\nIndividual fold R² scores: {cv_scores.round(4)}")
print(f"Mean R²: {cv_scores.mean():.4f}")
print(f"Std R²: {cv_scores.std():.4f}")
print(f"95% CI: [{cv_scores.mean() - 1.96*cv_scores.std():.4f}, {cv_scores.mean() + 1.96*cv_scores.std():.4f}]")

# Compare to single split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
single_split_r2 = r2_score(y_test, model.predict(X_test))
print(f"\nSingle train/test split R²: {single_split_r2:.4f}")
print(f"CV mean is {'higher' if cv_scores.mean() > single_split_r2 else 'lower'} than single split")

## Code: Visualizing Fold Variability

In [ ]:
from sklearn.model_selection import cross_validate

# Get more detailed CV results
cv_results = cross_validate(
    model, X_scaled, y, cv=10, 
    scoring=['r2', 'neg_mean_squared_error'],
    return_train_score=True
)

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² scores across folds
folds = range(1, 11)
axes[0].bar(folds, cv_results['test_r2'], alpha=0.7, label='Test', color='steelblue')
axes[0].bar(folds, cv_results['train_r2'], alpha=0.4, label='Train', color='orange')
axes[0].axhline(y=cv_results['test_r2'].mean(), color='red', linestyle='--', 
                label=f'Mean test R²={cv_results["test_r2"].mean():.3f}')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('R² Score')
axes[0].set_title('R² Scores Across 10 Folds')
axes[0].legend()
axes[0].set_xticks(folds)

# Distribution of scores
axes[1].hist(cv_results['test_r2'], bins=8, edgecolor='black', alpha=0.7)
axes[1].axvline(x=cv_results['test_r2'].mean(), color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('R² Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Test R² Scores')

plt.tight_layout()
plt.show()

print(f"Train-Test Gap: {cv_results['train_r2'].mean() - cv_results['test_r2'].mean():.4f}")
print("(Large gap may indicate overfitting)")

## Code: Stratified K-Fold for Classification

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Classification data
X = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['good_wine']
X_scaled = StandardScaler().fit_transform(X)

# Compare regular KFold vs StratifiedKFold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_model = LogisticRegression(random_state=42, max_iter=1000)

# Regular KFold
kfold_scores = cross_val_score(log_model, X_scaled, y, cv=kfold, scoring='roc_auc')

# Stratified KFold
stratified_scores = cross_val_score(log_model, X_scaled, y, cv=stratified_kfold, scoring='roc_auc')

print("KFOLD vs STRATIFIED KFOLD")
print("=" * 50)
print(f"\nRegular KFold:")
print(f"  Scores: {kfold_scores.round(4)}")
print(f"  Mean: {kfold_scores.mean():.4f} ± {kfold_scores.std():.4f}")

print(f"\nStratified KFold:")
print(f"  Scores: {stratified_scores.round(4)}")
print(f"  Mean: {stratified_scores.mean():.4f} ± {stratified_scores.std():.4f}")

print("\nStratified KFold typically has lower variance for imbalanced classes")

## Code: Cross-Validation for Model Selection

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

# Compare multiple models using CV
X = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['quality']
X_scaled = StandardScaler().fit_transform(X)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Lasso (α=0.1)': Lasso(alpha=0.1),
    'Ridge (α=10.0)': Ridge(alpha=10.0),
}

cv_comparison = []
for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='r2')
    cv_comparison.append({
        'Model': name,
        'Mean R²': scores.mean(),
        'Std R²': scores.std(),
        'Min R²': scores.min(),
        'Max R²': scores.max()
    })

cv_df = pd.DataFrame(cv_comparison)
print("MODEL COMPARISON VIA CROSS-VALIDATION")
print("=" * 60)
print(cv_df.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
x = range(len(cv_df))
plt.bar(x, cv_df['Mean R²'], yerr=cv_df['Std R²'], capsize=5, alpha=0.8)
plt.xticks(x, cv_df['Model'], rotation=15)
plt.ylabel('R² Score')
plt.title('Cross-Validation Model Comparison (Mean ± Std)')
plt.tight_layout()
plt.show()

## Expected Output & Interpretation

Cross-validation insights:

- **Score variability**: R² ranges ~0.33 to ~0.42 across folds
- **Reliable estimate**: CV mean is more stable than single split
- **Model ranking**: CV reveals which model consistently performs best
- **Stratified CV**: Reduces variance for imbalanced classification

## Hands-on Exercise 3.2

**Task**: Perform nested cross-validation.

1. Create an outer 5-fold CV loop
2. Within each outer fold, use 3-fold CV to select the best Ridge alpha from [0.01, 0.1, 1.0, 10.0]
3. Train the best model on the outer training fold and evaluate on outer test fold
4. Report the 5 outer fold test scores
5. Why is nested CV more reliable than simple CV for model selection?

**Time**: 15 minutes

In [ ]:
# TODO: nested CV - outer 5-fold; inner 3-fold GridSearchCV over Ridge alpha; report 5 outer scores
# Hint: from sklearn.model_selection import KFold, GridSearchCV; wrap StandardScaler+Ridge in a Pipeline


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X = wine_df.drop(columns=["quality", "good_wine"]); y = wine_df["quality"]

outer = KFold(n_splits=5, shuffle=True, random_state=42)
pipe = Pipeline([("sc", StandardScaler()), ("ridge", Ridge())])
grid = {"ridge__alpha": [0.01, 0.1, 1.0, 10.0]}

scores = []
for tr_idx, te_idx in outer.split(X):
    gs = GridSearchCV(pipe, grid, cv=3, scoring="r2")
    gs.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    scores.append(gs.score(X.iloc[te_idx], y.iloc[te_idx]))

print("Outer fold R2:", np.round(scores, 4))
print(f"Mean: {np.mean(scores):.4f} +/- {np.std(scores):.4f}")
print("Nested CV is more reliable because alpha is chosen INSIDE each outer fold, so the "
      "outer test folds never influence model selection -> no optimistic bias.")
```

</details>

# Module 3.3: Introduction to XGBoost

## Conceptual Overview

XGBoost (Extreme Gradient Boosting) is a powerful ensemble method that:

- Builds trees sequentially, each correcting previous errors
- Uses gradient descent to optimize a loss function
- Includes regularization to prevent overfitting
- Handles missing values automatically

XGBoost often wins machine learning competitions and works well on tabular data.

## Wow Factor Hook

You will train an XGBoost model that outperforms all previous models by a significant margin — and understand why it works so well.

## Code: Installing and Importing XGBoost

In [ ]:
# Install xgboost if needed (run in Colab)
# !pip install xgboost

import xgboost as xgb
from xgboost import XGBRegressor, XGBClassifier

print(f"XGBoost version: {xgb.__version__}")

# Prepare data
X = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['quality']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## Code: Basic XGBoost Model

In [ ]:
# Train basic XGBoost regressor
xgb_model = XGBRegressor(
    n_estimators=100,      # Number of trees
    max_depth=6,           # Maximum tree depth
    learning_rate=0.1,     # Step size shrinkage
    random_state=42,
    verbosity=0
)

xgb_model.fit(X_train, y_train)

# Predictions
y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

# Evaluate
print("XGBOOST REGRESSION RESULTS")
print("=" * 50)
print(f"\nTraining Performance:")
print(f"  R² Score: {r2_score(y_train, y_train_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")

print(f"\nTest Performance:")
print(f"  R² Score: {r2_score(y_test, y_test_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")

# Compare to Ridge baseline
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(StandardScaler().fit_transform(X_train), y_train)
ridge_r2 = r2_score(y_test, ridge_model.predict(StandardScaler().fit(X_train).transform(X_test)))
print(f"\nComparison - Ridge R²: {ridge_r2:.4f}")
print(f"XGBoost improvement: {((r2_score(y_test, y_test_pred) - ridge_r2) / ridge_r2 * 100):.1f}%")

## Code: Feature Importance in XGBoost

In [ ]:
# Get feature importances
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
axes[0].barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
axes[0].set_xlabel('Importance Score')
axes[0].set_title('XGBoost Feature Importances')
axes[0].invert_yaxis()

# Built-in XGBoost plot
xgb.plot_importance(xgb_model, ax=axes[1], max_num_features=10, importance_type='gain')
axes[1].set_title('XGBoost Feature Importance (Gain)')

plt.tight_layout()
plt.show()

print("Top 5 Most Important Features:")
print(importance_df.head().to_string(index=False))

## Code: Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

# Grid search with cross-validation
xgb_base = XGBRegressor(random_state=42, verbosity=0)

# Use smaller grid for demonstration (full grid takes time)
param_grid_small = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.2]
}

grid_search = GridSearchCV(
    xgb_base, 
    param_grid_small, 
    cv=3, 
    scoring='r2',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\nGRID SEARCH RESULTS")
print("=" * 50)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV R²: {grid_search.best_score_:.4f}")

# Evaluate best model
best_xgb = grid_search.best_estimator_
y_pred_best = best_xgb.predict(X_test)
print(f"Test R² (best model): {r2_score(y_test, y_pred_best):.4f}")

## Code: XGBoost for Classification

In [ ]:
# Classification: predict good wine
X = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['good_wine']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# XGBoost classifier with class imbalance handling
xgb_clf = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum(),  # Handle imbalance
    random_state=42,
    verbosity=0
)

xgb_clf.fit(X_train, y_train)

# Predictions
y_pred = xgb_clf.predict(X_test)
y_prob = xgb_clf.predict_proba(X_test)[:, 1]

print("XGBOOST CLASSIFICATION RESULTS")
print("=" * 50)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

# Compare to logistic regression
log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(StandardScaler().fit_transform(X_train), y_train)
log_prob = log_model.predict_proba(StandardScaler().fit(X_train).transform(X_test))[:, 1]
print(f"\nLogistic Regression ROC-AUC: {roc_auc_score(y_test, log_prob):.4f}")

## Code: Learning Curves for XGBoost

In [ ]:
# Training with early stopping to prevent overfitting
xgb_model_es = XGBRegressor(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    verbosity=0
)

# Split for early stopping validation
X_train_es, X_val, y_train_es, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

xgb_model_es.fit(
    X_train_es, y_train_es,
    eval_set=[(X_train_es, y_train_es), (X_val, y_val)],
    verbose=False
)

# Plot learning curves
results = xgb_model_es.evals_result()

plt.figure(figsize=(10, 6))
plt.plot(results['validation_0']['rmse'], label='Training')
plt.plot(results['validation_1']['rmse'], label='Validation')
plt.xlabel('Number of Trees')
plt.ylabel('RMSE')
plt.title('XGBoost Learning Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Observation: If validation error increases while training error decreases,")
print("the model is overfitting. Use early_stopping_rounds to prevent this.")

## Expected Output & Interpretation

XGBoost results:

- **Regression R² ~0.45-0.50**: Significant improvement over linear models (~0.36)
- **Classification ROC-AUC ~0.85**: Better than logistic regression (~0.82)
- **Feature importance**: Alcohol and volatile acidity remain top features
- **Hyperparameter tuning**: Can further improve by 2-5%

## Hands-on Exercise 3.3

**Task**: Tune an XGBoost model for optimal performance.

1. Train XGBoost with early stopping (early_stopping_rounds=20)
2. Use eval_metric='rmse' and monitor validation performance
3. Record the optimal number of trees (best_iteration)
4. Compare final test R² with and without early stopping
5. Which prevents overfitting better: fewer trees or early stopping?

**Time**: 15 minutes

In [ ]:
# TODO: XGBoost with early stopping (modern API); record best_iteration; compare test R^2 with/without
# Hint: in xgboost 2.x/3.x set early_stopping_rounds & eval_metric in the CONSTRUCTOR, pass eval_set to fit


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X = wine_df.drop(columns=["quality", "good_wine"]); y = wine_df["quality"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
X_t, X_v, y_t, y_v = train_test_split(X_tr, y_tr, test_size=0.2, random_state=42)

# Modern xgboost (>=2.0): early stopping is configured in the CONSTRUCTOR
es = XGBRegressor(n_estimators=500, max_depth=5, learning_rate=0.05,
                  early_stopping_rounds=20, eval_metric="rmse",
                  random_state=42, verbosity=0)
es.fit(X_t, y_t, eval_set=[(X_v, y_v)], verbose=False)
print("Best iteration (optimal #trees):", es.best_iteration)
print(f"Test R2 with early stopping:    {r2_score(y_te, es.predict(X_te)):.4f}")

full = XGBRegressor(n_estimators=500, max_depth=5, learning_rate=0.05,
                    random_state=42, verbosity=0).fit(X_tr, y_tr)
print(f"Test R2 without early stopping: {r2_score(y_te, full.predict(X_te)):.4f}")
print("Early stopping halts once validation error stops improving -> usually better than "
      "guessing a fixed, smaller tree count.")
```

</details>

# Module 3.4: Model Comparison Framework

## Conceptual Overview

Systematic model comparison requires:

- Consistent data splitting across all models
- Standardized evaluation metrics
- Multiple metrics for comprehensive assessment
- Cross-validation for reliable estimates
- Clear documentation of results

A comparison framework ensures fair and reproducible benchmarking.

## Wow Factor Hook

You will build a professional-grade comparison framework and generate a leaderboard ranking all models tried in this course.

## Code: Unified Comparison Framework

In [ ]:
from sklearn.pipeline import Pipeline

def evaluate_models(models, X, y, cv=5, task='regression'):
    """
    Evaluate multiple models using cross-validation.
    
    Parameters:
    - models: dict of {name: model} pairs
    - X, y: features and target
    - cv: number of cross-validation folds
    - task: 'regression' or 'classification'
    
    Returns:
    - DataFrame with evaluation results
    """
    
    results = []
    
    if task == 'regression':
        scorings = ['r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_error']
        metric_names = ['R²', 'RMSE', 'MAE']
    else:
        scorings = ['accuracy', 'roc_auc', 'f1']
        metric_names = ['Accuracy', 'ROC-AUC', 'F1']
    
    for name, model in models.items():
        print(f"Evaluating {name}...")
        
        row = {'Model': name}
        for scoring, metric_name in zip(scorings, metric_names):
            scores = cross_val_score(model, X, y, cv=cv, scoring=scoring)
            
            # Handle negative scores
            if 'neg_' in scoring:
                scores = -scores
            
            row[f'{metric_name} Mean'] = scores.mean()
            row[f'{metric_name} Std'] = scores.std()
        
        results.append(row)
    
    return pd.DataFrame(results)

print("Model comparison framework defined.")

## Code: Regression Model Comparison

In [ ]:
# Prepare data
X = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['quality']

# Scale for linear models
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define models to compare
regression_models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Ridge (α=10.0)': Ridge(alpha=10.0),
    'Lasso (α=0.1)': Lasso(alpha=0.1, max_iter=10000),
    'XGBoost (default)': XGBRegressor(random_state=42, verbosity=0),
    'XGBoost (tuned)': XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0)
}

# Note: Linear models use scaled data, XGBoost uses original
# For fair comparison, we'll create pipelines

from sklearn.pipeline import Pipeline

regression_models_pipeline = {
    'Linear Regression': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Ridge (α=1.0)': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Ridge (α=10.0)': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=10.0))]),
    'Lasso (α=0.1)': Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.1, max_iter=10000))]),
    'XGBoost (default)': XGBRegressor(random_state=42, verbosity=0),
    'XGBoost (tuned)': XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0)
}

# Run comparison
regression_results = evaluate_models(regression_models_pipeline, X, y, cv=5, task='regression')

print("\nREGRESSION MODEL COMPARISON")
print("=" * 80)
print(regression_results.round(4).to_string(index=False))

## Code: Classification Model Comparison

In [ ]:
# Classification data
X = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['good_wine']

# Define classification models
classification_models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), 
                                      ('model', LogisticRegression(random_state=42, max_iter=1000))]),
    'Logistic (C=0.1)': Pipeline([('scaler', StandardScaler()), 
                                   ('model', LogisticRegression(C=0.1, random_state=42, max_iter=1000))]),
    'XGBoost Classifier': XGBClassifier(n_estimators=100, max_depth=5, random_state=42, verbosity=0),
    'XGBoost (balanced)': XGBClassifier(n_estimators=100, max_depth=5, 
                                         scale_pos_weight=5, random_state=42, verbosity=0)
}

# Run comparison
classification_results = evaluate_models(classification_models, X, y, cv=5, task='classification')

print("\nCLASSIFICATION MODEL COMPARISON")
print("=" * 80)
print(classification_results.round(4).to_string(index=False))

## Code: Visual Model Comparison

In [ ]:
# Visualize regression results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² comparison
models_sorted = regression_results.sort_values('R² Mean', ascending=True)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(models_sorted)))
axes[0].barh(models_sorted['Model'], models_sorted['R² Mean'], 
             xerr=models_sorted['R² Std'], capsize=5, color=colors)
axes[0].set_xlabel('R² Score')
axes[0].set_title('Regression Models: R² Comparison')
axes[0].axvline(x=models_sorted['R² Mean'].max(), color='red', linestyle='--', alpha=0.7)

# RMSE comparison (lower is better)
models_sorted_rmse = regression_results.sort_values('RMSE Mean', ascending=False)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(models_sorted_rmse)))
axes[1].barh(models_sorted_rmse['Model'], models_sorted_rmse['RMSE Mean'],
             xerr=models_sorted_rmse['RMSE Std'], capsize=5, color=colors)
axes[1].set_xlabel('RMSE (Lower is Better)')
axes[1].set_title('Regression Models: RMSE Comparison')

plt.tight_layout()
plt.show()

## Expected Output & Interpretation

Comparison insights:

- **Clear winner**: XGBoost variants consistently outperform linear models
- **Tuning matters**: XGBoost (tuned) slightly better than default
- **Linear models**: Ridge and Lasso similar, both better than vanilla linear regression
- **Variability**: Standard deviations show result reliability

## Hands-on Exercise 3.4

**Task**: Add more models to the comparison.

1. Add `RandomForestRegressor` from sklearn.ensemble to the comparison
2. Add `GradientBoostingRegressor` from sklearn.ensemble
3. Run the comparison framework with these additional models
4. Update the leaderboard — where do they rank?
5. Which model would you recommend for production and why?

**Time**: 15 minutes

In [ ]:
# TODO: add RandomForestRegressor and GradientBoostingRegressor to evaluate_models(); re-rank the board
# Hint: reuse regression_models_pipeline and evaluate_models() from Module 3.4


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# reuse evaluate_models() and regression_models_pipeline from Module 3.4
extra = dict(regression_models_pipeline)
extra["Random Forest"] = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
extra["Gradient Boosting"] = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)

X = wine_df.drop(columns=["quality", "good_wine"]); y = wine_df["quality"]
board = evaluate_models(extra, X, y, cv=5, task="regression")
board = board.sort_values("R² Mean", ascending=False)
print(board.round(4).to_string(index=False))
print("Tree ensembles (RF, Gradient Boosting, XGBoost) usually top the board. For "
      "production, prefer the best CV score with low variance and reasonable cost.")
```

</details>

# Module 3.5: Final Benchmarking & Leaderboard

## Wow Factor: Production-Quality Benchmark Report

We will now generate a comprehensive benchmark report summarizing all models across the course.

## Code: Final Leaderboard

In [ ]:
# Comprehensive final benchmark
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# All models for final comparison
X = wine_df.drop(['quality', 'good_wine'], axis=1)
y = wine_df['quality']

final_models = {
    'Linear Regression': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Ridge Regression': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Lasso Regression': Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.1, max_iter=10000))]),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0)
}

# Evaluate all models
final_results = []
for name, model in final_models.items():
    r2_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    rmse_scores = -cross_val_score(model, X, y, cv=5, scoring='neg_root_mean_squared_error')
    
    final_results.append({
        'Rank': 0,  # Will assign later
        'Model': name,
        'R² Mean': r2_scores.mean(),
        'R² Std': r2_scores.std(),
        'RMSE Mean': rmse_scores.mean(),
        'RMSE Std': rmse_scores.std()
    })

# Create leaderboard
leaderboard = pd.DataFrame(final_results)
leaderboard = leaderboard.sort_values('R² Mean', ascending=False)
leaderboard['Rank'] = range(1, len(leaderboard) + 1)
leaderboard = leaderboard[['Rank', 'Model', 'R² Mean', 'R² Std', 'RMSE Mean', 'RMSE Std']]

print("=" * 80)
print("FINAL MODEL LEADERBOARD - WINE QUALITY PREDICTION")
print("=" * 80)
print(leaderboard.round(4).to_string(index=False))
print("=" * 80)

## Code: Leaderboard Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# R² Leaderboard
leaderboard_sorted = leaderboard.sort_values('R² Mean', ascending=True)
colors = ['gold' if i == len(leaderboard_sorted)-1 else 
          'silver' if i == len(leaderboard_sorted)-2 else 
          'chocolate' if i == len(leaderboard_sorted)-3 else 
          'steelblue' for i in range(len(leaderboard_sorted))]

bars = axes[0].barh(leaderboard_sorted['Model'], leaderboard_sorted['R² Mean'], 
                     xerr=leaderboard_sorted['R² Std'], capsize=5, color=colors)
axes[0].set_xlabel('R² Score', fontsize=12)
axes[0].set_title('Model Leaderboard: R² Score (Higher is Better)', fontsize=14)

# Add value labels
for bar, val in zip(bars, leaderboard_sorted['R² Mean']):
    axes[0].text(val + 0.02, bar.get_y() + bar.get_height()/2, 
                 f'{val:.3f}', va='center', fontsize=10)

# RMSE Leaderboard
leaderboard_sorted_rmse = leaderboard.sort_values('RMSE Mean', ascending=False)
bars2 = axes[1].barh(leaderboard_sorted_rmse['Model'], leaderboard_sorted_rmse['RMSE Mean'],
                      xerr=leaderboard_sorted_rmse['RMSE Std'], capsize=5, 
                      color=colors[::-1])  # Reverse colors (lower RMSE is better)
axes[1].set_xlabel('RMSE (Lower is Better)', fontsize=12)
axes[1].set_title('Model Leaderboard: RMSE', fontsize=14)

# Add value labels
for bar, val in zip(bars2, leaderboard_sorted_rmse['RMSE Mean']):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('model_leaderboard.png', dpi=150, bbox_inches='tight')
plt.show()

## Code: Winner Analysis

In [ ]:
# Analyze the winning model
winner_name = leaderboard.iloc[0]['Model']
winner_model = final_models[winner_name]

# Train on full data for final analysis
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

winner_model.fit(X_train, y_train)
y_pred = winner_model.predict(X_test)

print(f"WINNER ANALYSIS: {winner_name}")
print("=" * 60)
print(f"\nFinal Test Performance:")
print(f"  R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"  MAE: {mean_absolute_error(y_test, y_pred):.4f}")

# Feature importance
if hasattr(winner_model, 'feature_importances_'):
    importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': winner_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print(f"\nTop 5 Features:")
    for _, row in importance.head().iterrows():
        print(f"  {row['Feature']}: {row['Importance']:.4f}")

# Prediction analysis
print(f"\nPrediction Distribution:")
print(f"  Actual range: [{y_test.min()}, {y_test.max()}]")
print(f"  Predicted range: [{y_pred.min():.2f}, {y_pred.max():.2f}]")
print(f"  Predictions within ±0.5 of actual: {np.mean(np.abs(y_test - y_pred) <= 0.5)*100:.1f}%")
print(f"  Predictions within ±1.0 of actual: {np.mean(np.abs(y_test - y_pred) <= 1.0)*100:.1f}%")

## Code: Executive Summary Report

In [ ]:
print("=" * 80)
print("EXECUTIVE SUMMARY: WINE QUALITY PREDICTION PROJECT")
print("=" * 80)

print("""
OBJECTIVE:
  Predict wine quality (scale 3-8) from physicochemical properties

DATA:
  - 1,599 red wine samples
  - 15 features (11 original + 4 engineered)
  - Target: Quality rating (integer 3-8)

KEY FINDINGS:

1. FEATURE IMPORTANCE
   - Alcohol content is the strongest predictor of quality
   - Volatile acidity negatively impacts quality
   - Engineered features (acid_ratio, alcohol_density) added predictive value

2. MODEL PERFORMANCE PROGRESSION
   ┌─────────────────────┬──────────┬─────────────────────┐
   │ Stage               │ R² Score │ Key Improvement     │
   ├─────────────────────┼──────────┼─────────────────────┤
   │ Baseline (Linear)   │ ~0.36    │ Starting point      │
   │ + Feature Eng.      │ ~0.40    │ Domain knowledge    │
   │ + Regularization    │ ~0.41    │ Overfitting control │
   │ + XGBoost           │ ~0.48    │ Non-linear patterns │
   └─────────────────────┴──────────┴─────────────────────┘

3. WINNING MODEL: XGBoost
   - Best R² Score: ~0.48
   - RMSE: ~0.60 quality points
   - 75%+ predictions within 1 quality point

4. RECOMMENDATIONS
   - Use XGBoost for production deployment
   - Monitor alcohol and volatile acidity for quality control
   - Consider ensemble of top 3 models for robust predictions

5. LIMITATIONS
   - Class imbalance (few high/low quality wines)
   - R² ~0.48 suggests other unmeasured factors affect quality
   - Model trained on Portuguese red wines only
""")

print("=" * 80)

## Hands-on Exercise 3.5

**Task**: Create your own benchmark report.

1. Add at least one more model to the comparison (e.g., ElasticNet, SVM)
2. Create a final leaderboard with all models
3. Generate a visualization showing train vs test performance to detect overfitting
4. Write a 5-sentence executive summary of your findings
5. Which model would you recommend and why?

**Time**: 20 minutes

In [ ]:
# TODO: add ElasticNet and SVR; build a leaderboard (CV + train/test R^2); plot train vs test; 5-line summary
# Hint: from sklearn.svm import SVR; from sklearn.linear_model import ElasticNet


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.svm import SVR
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

X = wine_df.drop(columns=["quality", "good_wine"]); y = wine_df["quality"]
models = {
    "Ridge": Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))]),
    "ElasticNet": Pipeline([("sc", StandardScaler()),
                            ("m", ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=10000))]),
    "SVR (RBF)": Pipeline([("sc", StandardScaler()), ("m", SVR(C=1.0))]),
    "XGBoost": XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1,
                            random_state=42, verbosity=0),
}

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
rows = []
for name, m in models.items():
    cv = cross_val_score(m, X, y, cv=5, scoring="r2").mean()
    m.fit(X_tr, y_tr)
    rows.append({"Model": name, "CV R2": cv,
                 "Train R2": r2_score(y_tr, m.predict(X_tr)),
                 "Test R2": r2_score(y_te, m.predict(X_te))})

board = pd.DataFrame(rows).sort_values("CV R2", ascending=False)
board.insert(0, "Rank", range(1, len(board) + 1))
print(board.round(4).to_string(index=False))

x = np.arange(len(board))
plt.bar(x - 0.2, board["Train R2"], 0.4, label="Train")
plt.bar(x + 0.2, board["Test R2"], 0.4, label="Test")
plt.xticks(x, board["Model"], rotation=15); plt.legend(); plt.title("Train vs Test R2"); plt.show()
print("XGBoost leads on CV R2. SVR can overfit (large train-test gap); ElasticNet ~ Ridge. "
      "Recommend XGBoost for generalization, with Ridge kept as an interpretable baseline.")
```

</details>

# Module 3.6: Course Conclusion & Best Practices

## Course Summary

Over 10 hours, we covered the complete data mining lifecycle:

| Part | Topics | Key Skills |
|------|--------|------------|
| 1 | Data Understanding | Loading, EDA, Pandas, Preprocessing |
| 2 | Feature Engineering & Models | Engineering, Selection, Linear/Logistic, Regularization |
| 3 | Advanced Modeling | Metrics, CV, XGBoost, Benchmarking |

## Best Practices Checklist

In [ ]:
print("""
DATA MINING BEST PRACTICES CHECKLIST

□ DATA UNDERSTANDING
  ✓ Inspect shape, types, and missing values
  ✓ Visualize distributions and relationships
  ✓ Understand the target variable distribution
  ✓ Document data quality issues

□ PREPROCESSING
  ✓ Handle missing values appropriately
  ✓ Detect and treat outliers
  ✓ Scale features (after splitting!)
  ✓ Encode categorical variables

□ FEATURE ENGINEERING
  ✓ Create domain-relevant features
  ✓ Test polynomial and interaction terms
  ✓ Apply feature selection methods
  ✓ Document feature transformations

□ MODELING
  ✓ Start with simple baselines
  ✓ Use proper train/test splitting
  ✓ Apply cross-validation
  ✓ Tune hyperparameters systematically

□ EVALUATION
  ✓ Select metrics appropriate to the problem
  ✓ Report multiple metrics
  ✓ Use confidence intervals from CV
  ✓ Compare to meaningful baselines

□ DOCUMENTATION
  ✓ Record all experiments
  ✓ Version control code and data
  ✓ Create reproducible pipelines
  ✓ Write clear executive summaries
""")

## Common Pitfalls to Avoid

In [ ]:
print("""
TOP 10 DATA MINING MISTAKES

1. DATA LEAKAGE
   ❌ Scaling before splitting
   ✓ Always fit scalers on training data only

2. IGNORING CLASS IMBALANCE
   ❌ Using accuracy on imbalanced data
   ✓ Use F1, ROC-AUC, or balanced metrics

3. NO BASELINE
   ❌ Starting with complex models
   ✓ Establish simple baseline first

4. SINGLE TRAIN/TEST SPLIT
   ❌ Trusting one split
   ✓ Use cross-validation for reliable estimates

5. OVERFITTING
   ❌ Maximizing training performance
   ✓ Monitor train-test gap

6. WRONG METRIC
   ❌ Using R² for all problems
   ✓ Match metric to business objective

7. IGNORING FEATURE IMPORTANCE
   ❌ Black-box modeling
   ✓ Understand what drives predictions

8. NO REPRODUCIBILITY
   ❌ Random results
   ✓ Set random seeds everywhere

9. PREMATURE OPTIMIZATION
   ❌ Tuning before understanding data
   ✓ EDA first, then model

10. NO DOCUMENTATION
    ❌ Undocumented experiments
    ✓ Track all decisions and results
""")

## Final Code: Reusable Pipeline Template

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

def data_mining_pipeline(X, y, models, cv=5, task='regression'):
    """
    Complete data mining pipeline template.
    
    Parameters:
    - X: Feature matrix
    - y: Target vector
    - models: Dict of {name: model} pairs (can include pipelines)
    - cv: Number of cross-validation folds
    - task: 'regression' or 'classification'
    
    Returns:
    - Sorted leaderboard DataFrame
    """
    
    # Set scoring metrics
    if task == 'regression':
        primary_metric = 'r2'
        metric_name = 'R²'
    else:
        primary_metric = 'roc_auc'
        metric_name = 'ROC-AUC'
    
    # Evaluate all models
    results = []
    for name, model in models.items():
        scores = cross_val_score(model, X, y, cv=cv, scoring=primary_metric)
        results.append({
            'Model': name,
            f'{metric_name} Mean': scores.mean(),
            f'{metric_name} Std': scores.std()
        })
    
    # Create leaderboard
    leaderboard = pd.DataFrame(results)
    leaderboard = leaderboard.sort_values(f'{metric_name} Mean', ascending=False)
    leaderboard.insert(0, 'Rank', range(1, len(leaderboard) + 1))
    
    return leaderboard

# Example usage
print("Pipeline template defined. Example:")
print("leaderboard = data_mining_pipeline(X, y, models, cv=5, task='regression')")

## Next Steps for Continued Learning

Recommended topics to explore after this course:

1. **Deep Learning**: Neural networks for complex patterns
2. **Time Series**: Forecasting and temporal data
3. **Natural Language Processing**: Text data mining
4. **Computer Vision**: Image classification
5. **MLOps**: Model deployment and monitoring
6. **AutoML**: Automated machine learning pipelines

## Resources

- **Scikit-learn Documentation**: https://scikit-learn.org/stable/
- **XGBoost Documentation**: https://xgboost.readthedocs.io/
- **Pandas Documentation**: https://pandas.pydata.org/docs/
- **Kaggle**: https://www.kaggle.com/ (practice datasets and competitions)
- **Papers With Code**: https://paperswithcode.com/ (latest methods)

## Final Hands-on Challenge

**Capstone Exercise**: Apply the complete pipeline to a new dataset.

1. Load the Diabetes dataset from sklearn (`load_diabetes`)
2. Perform EDA and identify key features
3. Engineer at least 2 new features
4. Train and compare at least 4 different models
5. Generate a final leaderboard
6. Write a one-page summary report including:
   - Dataset description
   - Key EDA findings
   - Model comparison results
   - Recommendations

**Time**: 45-60 minutes (homework)

In [ ]:
# TODO: capstone on load_diabetes - EDA, >=2 engineered features, >=4 models, leaderboard, one-page summary
# Hint: from sklearn.datasets import load_diabetes; df = load_diabetes(as_frame=True).frame


<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
from sklearn.datasets import load_diabetes
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

df = load_diabetes(as_frame=True).frame.copy()

# 1-2. EDA
print(df.describe().round(3))
corr = (df.corr(numeric_only=True)["target"].drop("target")
          .sort_values(key=abs, ascending=False))
print("\nTop correlated features:")
print(corr.head().round(3))

# 3. engineer 2 features
df["bmi_bp"] = df["bmi"] * df["bp"]
df["s5_bmi"] = df["s5"] * df["bmi"]
X = df.drop(columns=["target"]); y = df["target"]

# 4. compare >= 4 models
models = {
    "Linear": Pipeline([("sc", StandardScaler()), ("m", LinearRegression())]),
    "Ridge": Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=1.0))]),
    "Lasso": Pipeline([("sc", StandardScaler()), ("m", Lasso(alpha=0.1, max_iter=10000))]),
    "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                            random_state=42, verbosity=0),
}
rows = []
for name, m in models.items():
    cv = cross_val_score(m, X, y, cv=5, scoring="r2")
    rows.append({"Model": name, "R² Mean": cv.mean(), "R² Std": cv.std()})

# 5. leaderboard
board = pd.DataFrame(rows).sort_values("R² Mean", ascending=False)
board.insert(0, "Rank", range(1, len(board) + 1))
print("\nLEADERBOARD")
print(board.round(4).to_string(index=False))

# 6. summary
print("""
Summary report
- Data: 442 diabetes patients, 10 baseline features predicting disease progression.
- EDA: BMI and serum measure s5 are the strongest individual predictors.
- Models: linear/Ridge are competitive; tree ensembles give modest gains at best.
- Recommendation: Ridge for interpretability + stability, XGBoost if peak accuracy matters.
""")
```

</details>

## Congratulations!

You have completed the **Applied Data Mining with Python** course.

Key accomplishments:
- Loaded, explored, and preprocessed real-world data
- Engineered features that improved model performance
- Built and evaluated multiple regression and classification models
- Applied regularization and advanced ensemble methods
- Created professional benchmark reports

**Certificate of Completion**: Data Mining Practitioner

## Thank You

Questions and feedback welcome.

Continue practicing with new datasets to solidify your skills.

**End of Course**